In [ ]:
import torch
import vitlab
from vitlab.activations import ActivationReader
from vitlab.datasets import get_splits
from vitlab.sae import load_layer_sae

DEVICE = "cuda"
BACKBONE_CKPT = "../runs/dinov2-base_full_epochs30/best"
SAE_CKPT = "../saes/dinov2_fitz_l9_topk16_d3072"
DATASET = "fitzpatrick17k"

model  = vitlab.load_model(BACKBONE_CKPT, device=DEVICE)
reader = ActivationReader(model.backbone)
sae = load_layer_sae(SAE_CKPT, device=DEVICE)

train, _, _ = get_splits(DATASET, model_key=model.spec.key)
px = train[0]["pixel_values"].unsqueeze(0).to(DEVICE)

In [ ]:
with torch.no_grad():
    acts    = reader.read(px, sae.spec.site)
    print(f"Activations Shape (Batch, Sequence, Dimension): {acts.shape}")
    patches = acts[:, model.spec.n_prefix_tokens:, :]
    print(f"Patches Shape (Batch, Patches, Dimension): {patches.shape}")
    codes   = sae.encode(patches.reshape(-1, patches.shape[-1]))
    print(f"Codes Shape (Batch x Sequence, SAE Dimension): {codes.shape}")

active = (codes > 0).sum(-1).float().mean()
print(f"{codes.shape[1]} concepts, {active:.1f} active per patch (should be ~top_k)")

In [ ]:
sae.spec

In [ ]:
import torch
import vitlab
from vitlab.datasets import get_splits
from vitlab.sae import load_bank                 # -> SAEBank keyed by site
from vitlab import viz
import vitlab.attribution as A

device = "cuda"
SITE = "blocks.9.resid_post"          # must match what the SAE was trained on
BACKBONE_CKPT = "../runs/dinov2-base_full_epochs30/best"
SAE_CKPT = "../saes/dinov2_fitz_l9_topk16_d3072"
DATASET = "fitzpatrick17k"

# 1. model (backbone + task head) and the SAE bank
model = vitlab.load_model(BACKBONE_CKPT, device=device)
bank  = load_bank({SITE: SAE_CKPT}, device=device)   # {site: sae_dir}

# 2. one image, preprocessed for this backbone; keep the batch dim
train, _, _ = get_splits(DATASET, model_key=model.spec.key)
image = train[0]["pixel_values"].unsqueeze(0)      # (1, 3, 224, 224)

# 3. two-stage attribution: patching screens -> ablation verifies
task = model.task_names[0]                         # or e.g. "fitzpatrick17k"


In [ ]:
image = train[0]["pixel_values"].unsqueeze(0)      # (1, 3, 224, 224)
r = A.attribution_patching(model, bank, image, SITE, task=task, top_k=10, device=device)

for c in r.top(5):
    print(f"#{c['rank']}  {c['site']}  F{c['feature']}  Δ={c['score']:.4f}")

# 4. figure
viz.use_style()
fig = viz.attribution_grid(image, r, model_key=model.spec.key, save="attr.pdf", show=True)

In [ ]:
_ = viz.attribution_stats(r, title="Attr Stats", show=True)

In [ ]:
share = A.token_group_attribution(model, image, task=task)
print(share.table())
_ = viz.token_group_trajectory(share, measure="ablation_drop", save="tg.pdf")

In [ ]:
import vitlab
import torch, warnings
from torch.utils.data import DataLoader, TensorDataset
from vitlab.sae import discover_bank
from vitlab import get_splits, make_loader 
from vitlab.circuits import (AttributionPatcher, CircuitDiscovery, compute_median_activations,
                             verify_edges, CircuitEvaluator, collect_class_images)
import vitlab.viz as V
V.use_style()

device = "cuda"
BACKBONE_CKPT = "../runs/dinov2-base_full_epochs30/best"
SAE_CKPT = "../saes/"
DATASET = "fitzpatrick17k"
TARGET_CLASS = 8

# 1. model (backbone + heads) and a bank with an SAE at EVERY layer
model = vitlab.load_model(BACKBONE_CKPT, device=device)
n_layers = model.spec.n_layers
bank = discover_bank(SAE_CKPT, device=device)   # walks the tree, keys by site
task = model.task_names[0]                                 # or task=DATASET explicitly

# 2. data: one loader for the median baseline (balanced), and class images to explain
train, _, _ = get_splits(DATASET, model_key=model.spec.key)
ref_loader = make_loader(train, batch_size=32, shuffle=True, num_workers=4)
class_images = collect_class_images(ref_loader, TARGET_CLASS, n_images=32, device=device)


In [ ]:
# 1. median baseline + discovery
medians = compute_median_activations(model, bank, ref_loader, n_layers, device=device, max_batches=20)
patcher = AttributionPatcher(model, bank, medians, task=task, device=device)
disco   = CircuitDiscovery(patcher, n_layers)               # layer set derived from bank
circ    = disco.discover_aggregated(class_images, TARGET_CLASS, top_k=5, use_libragrad=True)
circ.summary()
_ = verify_edges(model, bank, circ, class_images[0], task=task, device=device)
_ = circ.plot(save="circuit.pdf", show=True)



In [ ]:


# 3. evaluation: single circuit + AUC-over-k sweep
ev = CircuitEvaluator(model, bank, medians, target_class=TARGET_CLASS,
                      n_layers=n_layers, task=task, device=device)
res = ev.evaluate(class_images[0], circ)
print(f"faithfulness={res.faithfulness:.3f}  completeness={res.completeness:.3f}  causality={res.causality:.3f}")
results, auc_f, auc_c = ev.evaluate_over_k(class_images, disco)

# 4. all figures


#

In [ ]:
V.circuit_viz.circuit_metrics_curve(results, auc_faith=auc_f, auc_comp=auc_c,
                        max_features=1536, save="circuit_metrics.pdf")

In [ ]:
circ.to_html(model, bank, ref_loader, "circuit.html", model_key="dinov2-base")

In [ ]:
train.features["label"]